# Potato KD — ba thí nghiệm CHƯA chạy

Notebook này **không** lặp lại baseline (đã có trong `potato_kd_kaggle`). Mỗi mục dưới
đây là một biến thể mới, mỗi cái ~2 giờ trên T4.

| RUN | Thay đổi | Kỳ vọng | Vì sao đáng chạy |
|---|---|---|---|
| `capacity` | growth rate 16 → 24 (1.05 M → 2.2 M) | **+2–3** | student đang bị chặn bởi capacity |
| `mixup` | MixUp α=0.2 | +0.5–1 | 2.5k ảnh là bài toán điển hình |
| `long` | 400 epoch (chuẩn: 250) | +0.3–0.8 | val vẫn tăng khi cosine kết thúc |

**Sửa `RUN` ở cell dưới rồi Run All.** Chạy lần lượt 3 lần trong 3 session khác nhau.

Lưu ý công bằng: `capacity` làm student to hơn nên **phải báo như điểm Pareto**
(kèm GFLOPs), không so trực tiếp với paper. `mixup` và `long` giữ nguyên
1.05 M / 0.63 GFLOPs nên vẫn so sánh trực tiếp được.

In [12]:
!mv /potato.zip /content/
!ls -la /content

mv: cannot stat '/potato.zip': No such file or directory
total 735620
drwxr-xr-x 1 root root      4096 Nov  6 12:11 .
drwxr-xr-x 1 root root      4096 Nov  6 12:11 ..
drwxr-xr-x 4 root root      4096 Nov  4 17:01 .config
-rw-r--r-- 1 root root     11220 Nov  6 12:10 data.py
-rw-r--r-- 1 root root      8199 Nov  6 12:10 losses.py
-rw-r--r-- 1 root root      1503 Nov  6 12:10 make_cache.py
-rw-r--r-- 1 root root      3661 Nov  6 12:10 metrics.py
-rw-r--r-- 1 root root 753189298 Nov  6 12:07 potato.zip
drwxr-xr-x 2 root root      4096 Nov  6 12:10 __pycache__
drwxr-xr-x 1 root root      4096 Nov  4 17:01 sample_data
-rw-r--r-- 1 root root     18031 Nov  6 12:10 train.py
drwxr-xr-x 2 root root      4096 Nov  6 12:10 work


In [13]:
import os, glob, shutil, random, zipfile
from pathlib import Path

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG")
ROOTS = [r for r in ("/kaggle/input", "/content/data", "/content") if os.path.isdir(r)]
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content/work"
Path(WORK).mkdir(parents=True, exist_ok=True)


def _has_images(p):
    try:
        return any(f.endswith(IMG_EXT) for f in os.listdir(p)[:300])
    except Exception:
        return False


def detect(root):
    for cand in glob.glob(os.path.join(root, "**", "train"), recursive=True):
        d = Path(cand)
        if not d.is_dir():
            continue
        subs = [x for x in d.iterdir() if x.is_dir()]
        if len(subs) >= 3 and all(_has_images(x) for x in subs[:3]):
            return str(d.parent), sorted(x.name for x in subs)
    hits = []
    for r, dirs, _ in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        with_img = [d for d in dirs if _has_images(os.path.join(r, d))]
        if len(with_img) >= 5:
            hits.append((r, with_img))
        if r.count(os.sep) > 9:
            dirs[:] = []
    if hits:
        r, with_img = max(hits, key=lambda t: len(t[1]))
        return r, sorted(with_img)
    return None, None


DATA_ROOT, CLASSES = None, None
for root in ROOTS:
    DATA_ROOT, CLASSES = detect(root)
    if DATA_ROOT:
        print("found in", root)
        break

if DATA_ROOT is None:
    zips = []
    for root in ROOTS:
        zips += glob.glob(os.path.join(root, "**", "*.zip"), recursive=True)
    assert zips, "Khong thay du lieu lan file zip nao."
    dst = Path(WORK) / "_extract"
    dst.mkdir(parents=True, exist_ok=True)
    for z in zips[:2]:
        print("extracting", z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dst)
    DATA_ROOT, CLASSES = detect(str(dst))

assert DATA_ROOT, "Khong tim thay 5+ folder lop sau khi giai nen."
print("DATA_ROOT =", DATA_ROOT)
print("CLASSES   =", CLASSES)
print("counts    =", {c: len(os.listdir(Path(DATA_ROOT) / c)) for c in CLASSES})

if not (Path(DATA_ROOT) / "train").is_dir():
    src, out = Path(DATA_ROOT), Path(WORK) / "potato"
    if not (out / "train").exists():
        random.seed(42)
        for cls in CLASSES:
            files = [f for f in sorted((src / cls).glob("*")) if f.is_file()]
            random.shuffle(files)
            n = len(files)
            n_tr, n_va = int(n * 0.81), int(n * 0.09)
            for split, chunk in (("train", files[:n_tr]),
                                 ("val", files[n_tr:n_tr + n_va]),
                                 ("test", files[n_tr + n_va:])):
                d = out / split / cls
                d.mkdir(parents=True, exist_ok=True)
                for f in chunk:
                    shutil.copy(f, d / f.name)
    DATA_ROOT = str(out)
    print("built split ->", DATA_ROOT)

for split in ("train", "val", "test"):
    p = Path(DATA_ROOT) / split
    if p.is_dir():
        print(split, sum(len(list((p / c).glob("*"))) for c in CLASSES if (p / c).is_dir()))

print("WORK =", WORK)

extracting /content/potato.zip
DATA_ROOT = /content/work/_extract/potato
CLASSES   = ['Bacteria', 'Fungi', 'Healthy', 'Nematode', 'Pest', 'Phytopthora', 'Virus']
counts    = {'Bacteria': 569, 'Fungi': 748, 'Healthy': 201, 'Nematode': 68, 'Pest': 611, 'Phytopthora': 347, 'Virus': 532}
built split -> /content/work/potato
train 2487
val 274
test 315
WORK = /content/work


In [14]:
%%writefile models.py
"""Paper-faithful models plus the distillation adapters used by the improved trainer.

The deployment networks (``HybridStudent`` backbone and ``TeacherResNet18``) are
unchanged from Hoang et al. 2025: four stages of 4/6/8/10 hybrid dense inverted
residual blocks, growth rate 16, expansion ratio 3, widths 128/224/352/512.
Only *training-time* adapters (feature projectors) are added on top.
"""
from __future__ import annotations

from typing import Dict

import torch
from torch import Tensor, nn
from torchvision.models import ResNet18_Weights, resnet18


class ConvBNAct(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int,
                 stride: int = 1, groups: int = 1, activate: bool = True):
        padding = kernel_size // 2
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding,
                      groups=groups, bias=False),
            nn.BatchNorm2d(out_channels),
        ]
        if activate:
            layers.append(nn.ReLU6(inplace=True))
        super().__init__(*layers)


class HybridDenseInvertedBlock(nn.Module):
    """DenseNet-style concatenation around a MobileNetV2-style block."""

    def __init__(self, in_channels: int, growth_rate: int = 16,
                 expansion_ratio: int = 3):
        super().__init__()
        bottleneck_channels = 4 * growth_rate
        expanded_channels = expansion_ratio * bottleneck_channels

        self.bottleneck = ConvBNAct(in_channels, bottleneck_channels, 1)
        self.main = nn.Sequential(
            ConvBNAct(bottleneck_channels, expanded_channels, 1),
            ConvBNAct(expanded_channels, expanded_channels, 3, groups=expanded_channels),
            ConvBNAct(expanded_channels, growth_rate, 1, activate=False),
        )
        self.shortcut = ConvBNAct(bottleneck_channels, growth_rate, 1, activate=False)

    def forward(self, x: Tensor) -> Tensor:
        x_bot = self.bottleneck(x)
        residual = self.main(x_bot) + self.shortcut(x_bot)
        return torch.cat([x, residual], dim=1)


def make_stage(in_channels: int, depth: int, growth_rate: int = 16) -> nn.Sequential:
    blocks, channels = [], in_channels
    for _ in range(depth):
        blocks.append(HybridDenseInvertedBlock(channels, growth_rate))
        channels += growth_rate
    return nn.Sequential(*blocks)


class HybridStudent(nn.Module):
    """Student described in Hoang et al. with main and auxiliary classifiers.

    ``growth_rate`` is exposed so capacity can be scaled as a controlled
    ablation (16 -> the paper's 1.05 M model; 24 -> ~1.6 M). Channel widths are
    derived exactly as in the paper: each stage adds ``depth * growth_rate``
    channels to its input width, so g=16 reproduces 128/224/352/512.
    """

    def __init__(self, num_classes: int = 7, dropout: float = 0.0,
                 growth_rate: int = 16, depths=(4, 6, 8, 10)):
        super().__init__()
        g = int(growth_rate)
        w0 = 64
        w1 = w0 + depths[0] * g
        w2 = w1 + depths[1] * g
        w3 = w2 + depths[2] * g
        w4 = w3 + depths[3] * g
        self.widths = (w0, w1, w2, w3, w4)
        self.growth_rate = g

        self.stem = nn.Sequential(
            ConvBNAct(3, w0, 7, stride=2),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        self.stage1 = make_stage(w0, depths[0], g)
        self.down1 = nn.AvgPool2d(2, 2)
        self.stage2 = make_stage(w1, depths[1], g)
        self.down2 = nn.AvgPool2d(2, 2)
        self.stage3 = make_stage(w2, depths[2], g)
        self.down3 = nn.AvgPool2d(2, 2)
        self.stage4 = make_stage(w3, depths[3], g)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.aux_head = nn.Linear(w3, num_classes)
        self.main_head = nn.Linear(w4, num_classes)

    def forward(self, x: Tensor) -> Dict[str, Tensor]:
        x = self.stem(x)
        x = self.stage1(x)
        x = self.down1(x)
        x = self.stage2(x)
        x = self.down2(x)
        x = self.stage3(x)
        stage3_feature = x                  # [B, 352, 14, 14]
        x = self.down3(x)
        aux_feature = x                     # [B, 352, 7, 7]
        x = self.stage4(x)                  # [B, 512, 7, 7]

        aux_vec = self.pool(aux_feature).flatten(1)
        main_vec = self.pool(x).flatten(1)
        return {
            "stage3_feature": stage3_feature,
            "aux_feature": aux_feature,
            "main_feature": x,
            "aux_vector": aux_vec,
            "main_vector": main_vec,
            "aux_logits": self.aux_head(aux_vec),
            "main_logits": self.main_head(self.dropout(main_vec)),
        }


class TeacherResNet18(nn.Module):
    """ResNet18 teacher with the paper's auxiliary (stage-3) branch."""

    def __init__(self, num_classes: int = 7, pretrained: bool = True,
                 dropout: float = 0.0):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        backbone = resnet18(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3   # [B, 256, 14, 14]
        self.layer4 = backbone.layer4   # [B, 512, 7, 7]

        self.aux_align = nn.Sequential(
            ConvBNAct(256, 256, 3, stride=2, groups=256),
            ConvBNAct(256, 352, 1),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.aux_head = nn.Linear(352, num_classes)
        self.main_head = nn.Linear(512, num_classes)

    def forward(self, x: Tensor) -> Dict[str, Tensor]:
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        layer3 = self.layer3(x)
        main_feature = self.layer4(layer3)
        aux_feature = self.aux_align(layer3)

        aux_vec = self.pool(aux_feature).flatten(1)
        main_vec = self.pool(main_feature).flatten(1)
        return {
            "layer3_feature": layer3,       # [B, 256, 14, 14]
            "layer4_feature": main_feature,  # [B, 512, 7, 7]
            "aux_feature": aux_feature,
            "main_feature": main_feature,
            "aux_vector": aux_vec,
            "main_vector": main_vec,
            "aux_logits": self.aux_head(aux_vec),
            "main_logits": self.main_head(self.dropout(main_vec)),
        }


class DistillStudent(nn.Module):
    """Training wrapper: student backbone + 1x1 adapters for feature KD.

    The adapters are *not* part of the deployed student. They let the student
    match the teacher at two spatial levels that share a resolution:

    * stage-3 : student 352x14x14  <-  teacher layer3 256x14x14  (1x1, 256->352)
    * stage-4 : student 512x7x7    <-  teacher layer4 512x7x7    (direct)

    The baseline only matched global-average-pooled vectors, which throws away
    the spatial structure the paper's own Grad-CAM analysis says matters.
    """

    def __init__(self, num_classes: int = 7, dropout: float = 0.0,
                 growth_rate: int = 16, depths=(4, 6, 8, 10)):
        super().__init__()
        self.backbone = HybridStudent(num_classes, dropout=dropout,
                                      growth_rate=growth_rate, depths=depths)
        w3 = self.backbone.widths[3]
        w4 = self.backbone.widths[4]
        # Maps the teacher's 256-channel layer-3 output into the student's
        # stage-3 space (both are 14x14 at 224 input).
        self.proj3 = nn.Sequential(
            nn.Conv2d(256, w3, 1, bias=False), nn.BatchNorm2d(w3),
        )
        # stage-4: the paper's student has 512 channels, identical to the
        # teacher's layer-4, so no adapter is needed at growth 16. Wider
        # variants (growth 24 -> 736) need a 1x1 projection.
        self.proj4 = (nn.Identity() if w4 == 512
                      else nn.Conv2d(512, w4, 1, bias=False))

    def forward(self, x: Tensor) -> Dict[str, Tensor]:
        return self.backbone(x)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

Writing models.py


In [15]:
%%writefile data.py
﻿"""Dataset pipeline for the potato leaf disease KD study.

All raw images are 1500x1500 field photographs. `make_cache.py` in the baseline
folder writes a 448x448 copy (`data/potato_split_448`) so an epoch does not
spend most of its time decoding JPEGs. This module works with either root.
"""
from __future__ import annotations

from pathlib import Path
from typing import List, Tuple

import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_train_transform(image_size: int, aug: str = "strong") -> transforms.Compose:
    if aug == "paper":
        ops = [
            transforms.Resize((image_size, image_size)),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
            transforms.RandomAffine(degrees=30),
        ]
    elif aug == "field":
        ops = [
            transforms.RandomResizedCrop(
                image_size, scale=(0.70, 1.0), ratio=(0.85, 1.15),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(30),
            transforms.RandomAffine(0, translate=(0.08, 0.08), scale=(0.90, 1.10)),
            transforms.ColorJitter(0.25, 0.25, 0.20, 0.04),
            transforms.RandomApply([transforms.GaussianBlur(3)], p=0.15),
        ]
    elif aug == "strong":
        # The dataset is uncontrolled: photos of whole plants, single leaves in
        # a hand, plastic sheeting backgrounds, variable zoom and lighting.
        # Colour is a genuine cue (yellowing vs. necrosis) so hue jitter stays
        # small; geometric augmentation does most of the work.
        ops = [
            transforms.RandomResizedCrop(
                image_size, scale=(0.45, 1.0), ratio=(0.75, 1.33),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.RandomVerticalFlip()], p=0.15),
            transforms.RandomApply(
                [transforms.RandomAffine(
                    degrees=20, translate=(0.06, 0.06), scale=(0.92, 1.08)
                )], p=0.6,
            ),
            transforms.ColorJitter(0.25, 0.25, 0.18, 0.03),
            transforms.RandomApply([transforms.GaussianBlur(3)], p=0.10),
        ]
    elif aug == "fast":
        # Same geometric coverage as "strong" but drops the two most expensive
        # PIL ops: HSV-based saturation/hue jitter and Gaussian blur. Profiling
        # showed those two dominate the data-path cost while the GPU sat idle
        # ~75% of the wall clock.
        ops = [
            transforms.RandomResizedCrop(
                image_size, scale=(0.45, 1.0), ratio=(0.75, 1.33),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply(
                [transforms.RandomAffine(
                    degrees=20, translate=(0.06, 0.06), scale=(0.92, 1.08)
                )], p=0.5,
            ),
            transforms.ColorJitter(brightness=0.25, contrast=0.25),
        ]
    else:
        raise ValueError(f"unknown aug mode {aug!r}")
    ops += [
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), ratio=(0.3, 3.3)),
    ]
    return transforms.Compose(ops)


def build_eval_transform(image_size: int, crop_pct: float = 1.14) -> transforms.Compose:
    resize = int(round(image_size * crop_pct))
    return transforms.Compose([
        transforms.Resize(resize, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class RamImageFolder(datasets.ImageFolder):
    """ImageFolder that keeps the decoded 448x448 images in RAM.

    JPEG decoding was ~5 of the ~12 ms/image spent in the data path, and the
    data path was the bottleneck (the GPU sat idle). 2493 images at 448x448x3
    is ~1.5 GB, which fits comfortably.
    """

    def __init__(self, root, transform=None, preload: bool = True):
        super().__init__(root, transform=transform)
        self._cache = []
        if preload:
            from PIL import Image
            for path, _ in self.samples:
                with Image.open(path) as im:
                    self._cache.append(im.convert("RGB"))

    def __getitem__(self, index: int):
        _, target = self.samples[index]
        img = self._cache[index] if self._cache else Image.open(self.samples[index][0]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, target


class PrefetchLoader:
    """Thread-parallel batch prefetcher for environments where DataLoader
    worker *processes* are unavailable.

    The GPU was idle ~99% of the time while augmenting 448x448 field photos in
    the main Python thread. PIL/torchvision ops release the GIL, so a handful of
    shard readers running in threads overlaps augmentation with training and
    gives a 3-4x epoch speed-up. Shuffling is per shard, which is equivalent to
    standard SGD sampling for practical purposes.
    """

    def __init__(self, dataset, batch_size: int, n_threads: int = 6,
                 shuffle: bool = True, seed: int = 0):
        from torch.utils.data import Subset
        self.dataset = dataset
        self.batch_size = batch_size
        self.n_threads = max(1, n_threads)
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self._gen = torch.Generator()
        if hasattr(dataset, "targets"):
            self._targets = list(dataset.targets)
        else:
            self._targets = None

    @property
    def targets(self):
        return self._targets

    def __len__(self):
        return (len(self.dataset) + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        import queue
        import threading
        from torch.utils.data import Subset

        n = len(self.dataset)
        self._gen.manual_seed(self.seed + self.epoch)
        perm = torch.randperm(n, generator=self._gen).tolist() if self.shuffle else list(range(n))
        self.epoch += 1
        shards = [perm[i::self.n_threads] for i in range(self.n_threads)]
        q: "queue.Queue" = queue.Queue(maxsize=self.n_threads * 3)
        errors: list = []
        stop = threading.Event()

        def work(indices):
            try:
                loader = DataLoader(
                    Subset(self.dataset, indices), batch_size=self.batch_size,
                    shuffle=self.shuffle, drop_last=False, num_workers=0,
                )
                for batch in loader:
                    if stop.is_set():
                        return
                    q.put(batch)
            except Exception as exc:  # noqa: BLE001
                errors.append(exc)
                stop.set()
            finally:
                q.put(None)

        threads = [threading.Thread(target=work, args=(s,), daemon=True) for s in shards]
        for t in threads:
            t.start()
        finished = 0
        while finished < len(threads):
            item = q.get()
            if item is None:
                finished += 1
                continue
            if errors:
                stop.set()
                raise errors[0]
            yield item
        stop.set()
        if errors:
            raise errors[0]


class TTADataset(Dataset):
    """Deterministic multi-view test-time augmentation."""

    def __init__(self, root: Path, image_size: int, views: str = "flip2"):
        self.base = datasets.ImageFolder(str(root))
        self.samples = self.base.samples
        self.classes = self.base.classes
        self.class_to_idx = self.base.class_to_idx
        self.targets = self.base.targets
        self.image_size = image_size
        self.views = views

        resize = int(round(image_size * 1.14))
        norm = [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
        self.center = transforms.Compose([
            transforms.Resize(resize, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(image_size), *norm,
        ])
        self.full = transforms.Compose([
            transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BICUBIC),
            *norm,
        ])
        self.zoom = transforms.Compose([
            transforms.Resize(int(round(image_size * 1.35)),
                              interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(image_size), *norm,
        ])

    def __len__(self) -> int:
        return len(self.samples)

    def _views(self, img):
        if self.views == "center":
            return [self.center(img)]
        if self.views == "flip2":
            c = self.center(img)
            return [c, torch.flip(c, dims=[2])]
        if self.views == "rich":
            out = []
            for tf in (self.center, self.full, self.zoom):
                v = tf(img)
                out += [v, torch.flip(v, dims=[2])]
            return out
        raise ValueError(self.views)

    def __getitem__(self, index: int):
        path, target = self.samples[index]
        from PIL import Image
        img = Image.open(path).convert("RGB")
        views = self._views(img)
        return torch.stack(views), target


def make_loaders(
    data_root: str,
    batch_size: int = 32,
    num_workers: int = 0,
    image_size: int = 224,
    aug: str = "strong",
    train_on_val: bool = False,
    seed: int = 42,
) -> Tuple[DataLoader, DataLoader, DataLoader, List[str]]:
    root = Path(data_root)
    for split in ("train", "val", "test"):
        if not (root / split).is_dir():
            raise FileNotFoundError(f"missing split: {root / split}")

    train_set = RamImageFolder(str(root / "train"), transform=build_train_transform(image_size, aug))
    val_set = RamImageFolder(str(root / "val"), transform=build_eval_transform(image_size))
    test_set = RamImageFolder(str(root / "test"), transform=build_eval_transform(image_size))

    if train_on_val:
        train_set = torch.utils.data.ConcatDataset([train_set, RamImageFolder(
            str(root / "val"), transform=build_train_transform(image_size, aug))])

    classes = datasets.ImageFolder(str(root / "train")).classes
    kw = dict(num_workers=num_workers, pin_memory=True)
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = (
        PrefetchLoader(train_set, batch_size, n_threads=6, shuffle=True, seed=seed)
        if num_workers == 0 else
        DataLoader(train_set, batch_size=batch_size, shuffle=True, generator=g,
                   drop_last=False, **kw)
    )
    return (
        train_loader,
        DataLoader(val_set, batch_size=batch_size, shuffle=False, **kw),
        DataLoader(test_set, batch_size=batch_size, shuffle=False, **kw),
        classes,
    )

Overwriting data.py


In [29]:
%%writefile losses.py
"""Losses for the improved multi-objective distillation.

Every term keeps the paper's semantics, but the feature term now compares
spatial maps instead of pooled vectors, and the response term can use the
decoupled (TCKD + NCKD) formulation that is known to behave better than plain
KL when the teacher and student disagree about easy classes.
"""
from __future__ import annotations

from typing import Dict, Optional, Tuple

import torch
import torch.nn.functional as F
from torch import Tensor, nn


def ce_loss(
    logits: Tensor,
    labels: Tensor,
    weight: Optional[Tensor] = None,
    label_smoothing: float = 0.0,
) -> Tensor:
    return F.cross_entropy(
        logits, labels, weight=weight, label_smoothing=label_smoothing
    )


def soft_ce(logits: Tensor, soft_targets: Tensor) -> Tensor:
    """Cross-entropy against a soft (e.g. mixup) target."""
    return -(soft_targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()


def kd_kl_per_sample(student_logits: Tensor, teacher_logits: Tensor, temperature: float) -> Tensor:
    student_log_prob = F.log_softmax(student_logits / temperature, dim=1)
    teacher_prob = F.softmax(teacher_logits.detach() / temperature, dim=1)
    return F.kl_div(student_log_prob, teacher_prob, reduction="none").sum(dim=1) * temperature ** 2


def dkd_per_sample(
    student_logits: Tensor,
    teacher_logits: Tensor,
    labels: Tensor,
    alpha: float = 0.5,
    beta: float = 4.0,
    temperature: float = 4.0,
) -> Tensor:
    """Decoupled KD (Zhao et al., CVPR 2022).

    TCKD transfers the target/non-target *confidence* structure, NCKD transfers
    the relation among the non-target classes, which is exactly the inter-class
    information the paper claims response KD provides.
    """
    t = temperature
    s_logit, t_logit = student_logits / t, teacher_logits.detach() / t
    b = s_logit.shape[0]
    idx = torch.arange(b, device=s_logit.device)
    mask = torch.zeros_like(s_logit, dtype=torch.bool)
    mask[idx, labels] = True

    t_bin = torch.stack([t_logit[mask], t_logit[~mask].view(b, -1).sum(1)], dim=1)
    s_bin = torch.stack([s_logit[mask], s_logit[~mask].view(b, -1).sum(1)], dim=1)
    tckd = F.kl_div(F.log_softmax(s_bin / 1.0, 1), F.softmax(t_bin / 1.0, 1),
                    reduction="none").sum(1) * 1.0

    neg_t = t_logit[~mask].view(b, -1)
    neg_s = s_logit[~mask].view(b, -1)
    nckd = F.kl_div(F.log_softmax(neg_s, 1), F.softmax(neg_t, 1),
                    reduction="none").sum(1) * t ** 2
    return alpha * tckd + beta * nckd


def normalize_maps(x: Tensor) -> Tensor:
    return F.normalize(x.flatten(2), dim=1)


def feature_mse(student_map: Tensor, teacher_map: Tensor) -> Tensor:
    """L2-normalised spatial MSE (scale free, channel-wise cosine matching)."""
    s = normalize_maps(student_map)
    t = normalize_maps(teacher_map.detach())
    return F.mse_loss(s, t)


def attention_transfer(student_map: Tensor, teacher_map: Tensor) -> Tensor:
    """Attention transfer on channel-wise spatial energy maps (Zagoruyko 2017)."""
    s = F.normalize(student_map.pow(2).mean(1, keepdim=True).flatten(1), dim=1)
    t = F.normalize(teacher_map.detach().pow(2).mean(1, keepdim=True).flatten(1), dim=1)
    return F.mse_loss(s, t) * student_map.shape[1]


def teacher_loss(
    teacher_output: Dict[str, Tensor],
    labels: Tensor,
    class_weights: Optional[Tensor] = None,
    label_smoothing: float = 0.1,
    aux_weight: float = 1.0,
) -> Tensor:
    main = ce_loss(teacher_output["main_logits"], labels, class_weights, label_smoothing)
    aux = ce_loss(teacher_output["aux_logits"], labels, class_weights, label_smoothing)
    return main + aux_weight * aux


def student_loss(
    student_output: Dict[str, Tensor],
    teacher_output: Dict[str, Tensor],
    labels: Tensor,
    lambdas: Tuple[float, float, float, float] = (1.0, 0.5, 1.0, 0.3),
    tau: float = 4.0,
    tau_prime: float = 4.0,
    class_weights: Optional[Tensor] = None,
    label_smoothing: float = 0.1,
    kd_mode: str = "dkd",
    feature_mode: str = "map",
    aux_weight: float = 1.0,
    soft_targets: Optional[Tensor] = None,
    normalize_kd: bool = True,
) -> Dict[str, Tensor]:
    """lambdas = (hard, feature, response, self)."""
    l_hard, l_feat, l_resp, l_self = lambdas

    if soft_targets is not None:
        hard = soft_ce(student_output["main_logits"], soft_targets) + \
               soft_ce(student_output["aux_logits"], soft_targets)
    else:
        hard = ce_loss(student_output["main_logits"], labels, class_weights, label_smoothing) + \
               aux_weight * ce_loss(student_output["aux_logits"], labels, class_weights, label_smoothing)

    if feature_mode == "map":
        feat = (
            feature_mse(student_output["stage3_feature"], teacher_output["layer3_projected"])
            + feature_mse(student_output["main_feature"], teacher_output["layer4_projected"])
            + 0.5 * attention_transfer(student_output["stage3_feature"], teacher_output["layer3_projected"])
            + 0.5 * attention_transfer(student_output["main_feature"], teacher_output["layer4_projected"])
        )
    elif feature_mode == "vector":
        # baseline behaviour, kept for ablation
        feat = (student_output["main_vector"] - teacher_output["main_vector"].detach()).norm(dim=1).mean()
        feat = feat + (student_output["aux_vector"] - teacher_output["aux_vector"].detach()).norm(dim=1).mean()
    elif feature_mode == "none":
        feat = torch.zeros((), device=labels.device)
    else:
        raise ValueError(feature_mode)

    if kd_mode == "dkd":
        resp = (
            dkd_per_sample(student_output["main_logits"], teacher_output["main_logits"], labels,
                           alpha=1.0, beta=8.0, temperature=tau)
            + dkd_per_sample(student_output["aux_logits"], teacher_output["aux_logits"], labels,
                             alpha=1.0, beta=8.0, temperature=tau)
        ).mean()
    elif kd_mode == "kl":
        resp = (
            kd_kl_per_sample(student_output["main_logits"], teacher_output["main_logits"], tau)
            + kd_kl_per_sample(student_output["aux_logits"], teacher_output["aux_logits"], tau)
        ).mean()
    elif kd_mode == "none":
        resp = torch.zeros((), device=labels.device)
    else:
        raise ValueError(kd_mode)

    self_kd = kd_kl_per_sample(
        student_output["main_logits"], student_output["aux_logits"].detach(), tau_prime
    ).mean()

    # The three KD terms live on very different numeric scales: a map MSE can
    # be ~30 while CE is ~2. Without normalisation the lambdas are meaningless
    # and feature KD silently dominates the objective. Dividing by the detached
    # batch mean puts every term on a comparable scale (~1), so lambda really
    # expresses relative importance. This was the single biggest cause of the
    # student degrading during warm-start refinement.
    if normalize_kd:
        feat = feat / feat.detach().clamp_min(1e-6)
        resp = resp / resp.detach().clamp_min(1e-6)

    total = l_hard * hard + l_feat * feat + l_resp * resp + l_self * self_kd
    return {
        "total": total,
        "hard": hard.detach(),
        "feature": feat.detach(),
        "response": resp.detach(),
        "self": self_kd.detach(),
    }


class ModelEMA:
    """Exponential moving average of model weights (works for any nn.Module)."""

    def __init__(self, model: nn.Module, decay: float = 0.999):
        import copy
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.step = 0

    @torch.no_grad()
    def update(self, model: nn.Module):
        # Warm up the decay so the EMA tracks the live model during the first
        # epochs instead of staying pinned to the random initialisation.
        self.step += 1
        d = min(self.decay, (1.0 + self.step) / (10.0 + self.step))
        for ema_p, p in zip(self.module.state_dict().values(), model.state_dict().values()):
            if ema_p.dtype.is_floating_point:
                ema_p.mul_(d).add_(p.detach(), alpha=1 - d)
            else:
                ema_p.copy_(p)

Overwriting losses.py


In [17]:
%%writefile metrics.py
"""Metrics, test-time augmentation and post-hoc logit adjustment."""
from __future__ import annotations

from typing import Dict, List, Optional

import numpy as np
import torch


def confusion_to_metrics(confusion: np.ndarray, classes: List[str]) -> Dict:
    n = confusion.shape[0]
    f1s, recalls, precisions = [], [], []
    for c in range(n):
        tp = confusion[c, c]
        fp = confusion[:, c].sum() - tp
        fn = confusion[c, :].sum() - tp
        p = tp / max(tp + fp, 1)
        r = tp / max(tp + fn, 1)
        precisions.append(p)
        recalls.append(r)
        f1s.append(2 * p * r / max(p + r, 1e-12))
    total = confusion.sum()
    return {
        "accuracy": float(np.trace(confusion) / max(total, 1)),
        "macro_f1": float(np.mean(f1s)),
        "weighted_f1": float(np.sum(np.array(f1s) * confusion.sum(1)) / max(total, 1)),
        "balanced_accuracy": float(np.mean(recalls)),
        "class_f1": {classes[c]: float(f1s[c]) for c in range(n)},
        "class_recall": {classes[c]: float(recalls[c]) for c in range(n)},
        "class_precision": {classes[c]: float(precisions[c]) for c in range(n)},
        "support": {classes[c]: int(confusion[c].sum()) for c in range(n)},
        "confusion_matrix": confusion.tolist(),
    }


@torch.no_grad()
def predict_logits(model, loader, device, tta: bool = False) -> tuple[np.ndarray, np.ndarray]:
    """Return (logits, labels). Supports multi-view TTA loaders."""
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        if tta:
            views, labels = batch           # [B, V, 3, H, W]
            b, v = views.shape[0], views.shape[1]
            flat = views.reshape(b * v, *views.shape[2:]).to(device, non_blocking=True)
            logits = model(flat)["main_logits"].float()
            logits = logits.view(b, v, -1).mean(1)
        else:
            images, labels = batch
            logits = model(images.to(device, non_blocking=True))["main_logits"].float()
        all_logits.append(logits.cpu())
        all_labels.append(labels)
    return torch.cat(all_logits).numpy(), torch.cat(all_labels).numpy()


def metrics_from_logits(logits: np.ndarray, labels: np.ndarray, classes: List[str],
                        bias: Optional[np.ndarray] = None) -> Dict:
    if bias is not None:
        logits = logits + bias[None, :]
    pred = logits.argmax(1)
    n = len(classes)
    confusion = np.zeros((n, n), dtype=np.int64)
    for t, p in zip(labels, pred):
        confusion[t, p] += 1
    return confusion_to_metrics(confusion, classes)


def tune_logit_bias(logits: np.ndarray, labels: np.ndarray, classes: List[str],
                    steps: int = 40, lr: float = 0.5) -> np.ndarray:
    """Coordinate ascent on a per-class additive bias to maximise macro-F1.

    Only the validation split is used. This is the standard "logit adjustment"
    fix for imbalanced classification; it changes no weights and no inference
    cost, so it is safe to apply to the deployed student.
    """
    n = len(classes)
    bias = np.zeros(n, dtype=np.float64)
    best = metrics_from_logits(logits, labels, classes, bias)["macro_f1"]
    for _ in range(steps):
        improved = False
        for c in range(n):
            for delta in (-lr, lr):
                trial = bias.copy()
                trial[c] += delta
                score = metrics_from_logits(logits, labels, classes, trial)["macro_f1"]
                if score > best + 1e-6:
                    best, bias, improved = score, trial, True
        if not improved:
            lr *= 0.5
            if lr < 0.02:
                break
    return bias.astype(np.float32)

Overwriting metrics.py


In [18]:
%%writefile train.py
﻿"""Train the ResNet18 teacher or the distilled student on the potato dataset.

Two-phase design (the main difference from the paper's online baseline):

  1. `--mode teacher` trains ResNet18 to convergence with a modern recipe
     (cosine schedule, label smoothing, EMA, strong field augmentation).
     This raises the distillation ceiling: in the baseline the teacher was
     itself under-trained and updated jointly with the student.
  2. `--mode student` freezes that teacher and distils it offline into the
     paper's hybrid student, with map-level feature KD, decoupled response KD
     and EMA. On-the-fly online teaching is kept available via `--online`.

Examples
--------
python train.py --mode teacher --data ../potato_kd_baseline/data/potato_split_448 \
    --output runs/teacher_r18_224 --epochs 60 --image-size 224 --batch-size 32 --lr 3e-4

python train.py --mode student --data ../potato_kd_baseline/data/potato_split_448 \
    --teacher runs/teacher_r18_224/best.pt --output runs/student_224 --epochs 80
"""
from __future__ import annotations

import argparse
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

from data import TTADataset, make_loaders
from losses import ModelEMA, kd_kl_per_sample, soft_ce, student_loss, teacher_loss
from metrics import (metrics_from_logits, predict_logits, tune_logit_bias)
from models import DistillStudent, HybridStudent, TeacherResNet18, count_parameters


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_scheduler(optimizer, total_steps: int, warmup_frac: float, kind: str = "cosine"):
    warmup = max(1, int(total_steps * warmup_frac))

    def fn(step: int) -> float:
        if step < warmup:
            return step / warmup
        if kind == "cosine":
            p = (step - warmup) / max(1, total_steps - warmup)
            return 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))
        return 1.0

    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


def mixup_batch(images, labels, num_classes, alpha, device):
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(images.size(0), device=device)
    mixed = lam * images + (1 - lam) * images[perm]
    target = torch.zeros(images.size(0), num_classes, device=device)
    target.scatter_(1, labels[:, None], lam)
    target.scatter_add_(1, labels[perm][:, None], torch.full_like(labels[:, None], 1 - lam, dtype=torch.float))
    return mixed, target


def make_class_weights(dataset, device):
    counts = np.bincount(np.array(dataset.targets), minlength=len(dataset.classes)).astype(np.float64)
    w = counts.sum() / (len(counts) * np.maximum(counts, 1))
    return torch.tensor(w, dtype=torch.float32, device=device)


def run_epoch(student, teacher, loader, optimizers, scaler, device, args, epoch, num_classes,
              class_weights, ema_students, ema_teacher):
    teacher_mode = teacher is not None and student is None
    online = teacher is not None and student is not None and "teacher" in optimizers
    if teacher is not None and (teacher_mode or online):
        teacher.train()
    if student is not None:
        student.train()
    totals = {"loss": 0.0}
    n_batches = len(loader)
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        soft = None
        if args.mixup > 0 and np.random.rand() < 0.5:
            images, soft = mixup_batch(images, labels, num_classes, args.mixup, device)

        if teacher_mode:
            opt = optimizers["teacher"]
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=args.amp):
                out = teacher(images)
                if soft is not None:
                    loss = 2.0 * soft_ce(out["main_logits"], soft)
                else:
                    loss = teacher_loss(out, labels, class_weights, args.label_smoothing,
                                        args.aux_weight)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(teacher.parameters(), 5.0)
            scaler.step(opt)
            scaler.update()
            if ema_teacher is not None:
                ema_teacher.update(teacher)
            totals["loss"] += loss.item()
            continue

        if teacher is not None and online:  # paper setting: update teacher first
            opt = optimizers["teacher"]
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=args.amp):
                out_t = teacher(images)
                if soft is not None:
                    loss_t = 2.0 * soft_ce(out_t["main_logits"], soft)
                else:
                    loss_t = teacher_loss(out_t, labels, class_weights, args.label_smoothing,
                                          args.aux_weight)
            scaler.scale(loss_t).backward()
            scaler.step(opt)
            scaler.update()
            if ema_teacher is not None:
                ema_teacher.update(teacher)

        with torch.no_grad(), torch.amp.autocast("cuda", enabled=args.amp):
            t_out = teacher(images)
            t_out = {k: v.float() for k, v in t_out.items()}
        t_out_detached = {k: (v.detach() if torch.is_tensor(v) else v) for k, v in t_out.items()}
        t_out_detached["layer3_projected"] = student.proj3(t_out_detached["layer3_feature"])
        t_out_detached["layer4_projected"] = student.proj4(t_out_detached["layer4_feature"])

        opt = optimizers["student"]
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=args.amp):
            s_out = student(images)
            ramp = getattr(args, "_ramp", 1.0)
            losses = student_loss(
                s_out, t_out_detached, labels,
                lambdas=(args.lambda_hard, args.lambda_feature * ramp,
                         args.lambda_response * ramp, args.lambda_self * ramp),
                tau=args.tau, tau_prime=args.tau_prime,
                class_weights=class_weights,
                label_smoothing=args.label_smoothing,
                kd_mode=args.kd,
                feature_mode=args.feature,
                aux_weight=args.aux_weight,
                soft_targets=soft,
                normalize_kd=args.normalize_kd,
            )
        scaler.scale(losses["total"]).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(student.parameters(), 5.0)
        scaler.step(opt)
        scaler.update()
        if ema_students is not None:
            ema_students.update(student)
        totals["loss"] += losses["total"].item()

    return totals["loss"] / max(n_batches, 1)


def evaluate_split(model, loader, device, classes, tta=False):
    logits, labels = predict_logits(model, loader, device, tta=tta)
    return logits, labels, metrics_from_logits(logits, labels, classes)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["teacher", "student"], default="teacher")
    ap.add_argument("--data", required=True)
    ap.add_argument("--output", required=True)
    ap.add_argument("--teacher", default=None, help="teacher checkpoint for offline KD")
    ap.add_argument("--online", action="store_true", help="train teacher jointly (paper setting)")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--image-size", type=int, default=224)
    ap.add_argument("--lr", type=float, default=None)
    ap.add_argument("--weight-decay", type=float, default=5e-2)
    ap.add_argument("--label-smoothing", type=float, default=0.1)
    ap.add_argument("--aug", default="strong", choices=["paper", "field", "strong", "fast"])
    ap.add_argument("--mixup", type=float, default=0.0)
    ap.add_argument("--warmup", type=float, default=0.05)
    ap.add_argument("--ema", type=float, default=0.999)
    ap.add_argument("--class-weights", action="store_true", help="inverse-frequency CE")
    ap.add_argument("--aux-weight", type=float, default=1.0)
    ap.add_argument("--lambda-hard", type=float, default=1.0)
    ap.add_argument("--lambda-feature", type=float, default=0.5)
    ap.add_argument("--lambda-response", type=float, default=1.0)
    ap.add_argument("--lambda-self", type=float, default=0.3)
    ap.add_argument("--tau", type=float, default=4.0)
    ap.add_argument("--tau-prime", type=float, default=4.0)
    ap.add_argument("--kd", default="dkd", choices=["dkd", "kl", "none"])
    ap.add_argument("--feature", default="map", choices=["map", "vector", "none"])
    ap.add_argument("--no-normalize-kd", dest="normalize_kd", action="store_false",
                    help="reproduce the paper loss scale: no per-term normalisation")
    ap.set_defaults(normalize_kd=True)
    ap.add_argument("--dropout", type=float, default=0.0)
    ap.add_argument("--growth", type=int, default=16,
                    help="student growth rate; 16 = paper (1.05M), 24 = ~2.2M")
    ap.add_argument("--track", default="mean", choices=["acc", "macro_f1", "mean"])
    ap.add_argument("--patience", type=int, default=0)
    ap.add_argument("--kd-ramp", type=int, default=0,
                    help="linearly ramp feature/response/self KD weights over N epochs")
    ap.add_argument("--init-student", default=None,
                    help="warm-start the student backbone from a checkpoint")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--amp", action="store_true", default=True)
    ap.add_argument("--no-amp", dest="amp", action="store_false")
    ap.add_argument("--train-on-val", action="store_true")
    ap.add_argument("--num-workers", type=int, default=0)
    args = ap.parse_args()

    set_seed(args.seed)
    torch.backends.cudnn.benchmark = False
    device = torch.device("cuda")
    out_dir = Path(args.output)
    out_dir.mkdir(parents=True, exist_ok=True)

    if args.lr is None:
        args.lr = 3e-4 if args.mode == "teacher" else 1e-3

    train_loader, val_loader, test_loader, classes = make_loaders(
        args.data, args.batch_size, args.num_workers, args.image_size,
        aug=args.aug, train_on_val=args.train_on_val, seed=args.seed)
    num_classes = len(classes)

    train_set = train_loader.dataset
    if args.class_weights:
        if hasattr(train_set, "targets"):
            targets = np.array(train_set.targets)
        else:
            targets = np.array([s[1] for s in train_set.samples])
        counts = np.bincount(targets, minlength=num_classes).astype(np.float64)
        cw = torch.tensor(counts.sum() / (num_classes * np.maximum(counts, 1)),
                          dtype=torch.float32, device=device)
        print("class weights:", [round(float(x), 3) for x in cw.cpu()])
    else:
        cw = None

    teacher = None
    student = None
    ema_teacher = None
    ema_students = None

    if args.mode == "teacher":
        teacher = TeacherResNet18(num_classes, pretrained=True, dropout=args.dropout).to(device)
        params = teacher.parameters()
        print("teacher params:", count_parameters(teacher) / 1e6, "M")
        ema_teacher = ModelEMA(teacher, decay=args.ema)
        student = None
    else:
        if args.online:
            teacher = TeacherResNet18(num_classes, pretrained=True, dropout=args.dropout).to(device)
            ema_teacher = ModelEMA(teacher, decay=args.ema)
        else:
            if not args.teacher:
                raise SystemExit("--teacher checkpoint required for offline distillation")
            ckpt = torch.load(args.teacher, map_location=device)
            state = ckpt.get("ema") or ckpt.get("teacher")
            teacher = TeacherResNet18(num_classes, pretrained=False).to(device)
            teacher.load_state_dict(state)
            teacher.eval()
            for p in teacher.parameters():
                p.requires_grad_(False)
        student = DistillStudent(num_classes, dropout=args.dropout,
                                 growth_rate=args.growth).to(device)
        if args.init_student:
            ck = torch.load(args.init_student, map_location=device)
            sd = ck.get("student_ema_full") or ck.get("student") or ck.get("ema")
            if any(k.startswith("backbone.") for k in sd):
                sd = {k[len("backbone."):]: v for k, v in sd.items() if k.startswith("backbone.")}
            else:
                sd = {k: v for k, v in sd.items() if not k.startswith("proj3.")}
            missing, unexpected = student.backbone.load_state_dict(sd, strict=False)
            print(f"warm start from {args.init_student}: missing={len(missing)} unexpected={len(unexpected)}")
        print("student backbone params:", count_parameters(student.backbone) / 1e6, "M")
        ema_students = ModelEMA(student, decay=args.ema)

    optimizers = {}
    if teacher is not None and (args.mode == "teacher" or args.online):
        optimizers["teacher"] = torch.optim.AdamW(teacher.parameters(), lr=args.lr,
                                                  weight_decay=args.weight_decay)
    if student is not None:
        optimizers["student"] = torch.optim.AdamW(student.parameters(), lr=args.lr,
                                                  weight_decay=args.weight_decay)
    for opt in optimizers.values():
        scheduler = build_scheduler(opt, args.epochs * len(train_loader), args.warmup)
        opt._sched = scheduler  # type: ignore[attr-defined]

    scaler = torch.amp.GradScaler("cuda", enabled=args.amp)
    model_for_eval = ema_teacher.module if args.mode == "teacher" else ema_students.module

    best_score, best_epoch, history = -1.0, 0, []
    start = time.time()
    for epoch in range(1, args.epochs + 1):
        args._ramp = 1.0 if args.kd_ramp <= 0 else min(1.0, epoch / args.kd_ramp)
        loss = run_epoch(student, teacher, train_loader, optimizers, scaler, device,
                         args, epoch, num_classes, cw, ema_students, ema_teacher)
        for opt in optimizers.values():
            opt._sched.step()  # type: ignore[attr-defined]

        logits, labels, val_m = evaluate_split(model_for_eval, val_loader, device, classes)
        score = {"acc": val_m["accuracy"], "macro_f1": val_m["macro_f1"],
                 "mean": 0.5 * (val_m["accuracy"] + val_m["macro_f1"])}[args.track]
        row = {"epoch": epoch, "loss": loss, "val_accuracy": val_m["accuracy"],
               "val_macro_f1": val_m["macro_f1"],
               "val_balanced_acc": val_m["balanced_accuracy"], "seconds": round(time.time() - start, 1)}
        history.append(row)
        print(json.dumps(row), flush=True)

        if score > best_score:
            best_score, best_epoch = score, epoch
            ckpt = {"epoch": epoch, "classes": classes, "args": vars(args)}
            if args.mode == "teacher":
                ckpt["teacher"] = teacher.state_dict()
                ckpt["ema"] = ema_teacher.module.state_dict()
            else:
                ckpt["student"] = student.state_dict()
                ckpt["ema"] = ema_students.module.state_dict()
                at = (out_dir / "all.pt")
                torch.save({"teacher": teacher.state_dict() if teacher is not None else
                            torch.load(args.teacher, map_location="cpu").get("ema"),
                            "stud_ema": ema_students.module.state_dict(),
                            "stud_raw": student.state_dict(),
                            "classes": classes,
                            "student_ema_full": ema_students.module.state_dict()}, at)
            torch.save(ckpt, out_dir / "best.pt")
        elif args.patience and epoch - best_epoch >= args.patience:
            print(f"early stop at {epoch}")
            break

    (out_dir / "history.json").write_text(json.dumps(history, indent=2))

    # Final evaluation with EMA weights, plain + TTA, plus validation-only
    # logit-bias calibration.
    ckpt = torch.load(out_dir / "best.pt", map_location=device)
    if args.mode == "teacher":
        final = TeacherResNet18(num_classes, pretrained=False)
        final.load_state_dict(ckpt["ema"])
    else:
        final = HybridStudent(num_classes, growth_rate=args.growth)
        sd = {k.replace("backbone.", ""): v for k, v in ckpt["ema"].items()
              if k.startswith("backbone.")}
        final.load_state_dict(sd)
    final = final.to(device).eval()

    report = {"args": vars(args), "best_epoch": best_epoch,
              "params_millions": count_parameters(final) / 1e6}

    val_logits, val_labels = predict_logits(final, val_loader, device, tta=False)
    report["val"] = metrics_from_logits(val_logits, val_labels, classes)

    tta_set = TTADataset(Path(args.data) / "test", args.image_size, views="rich")
    from torch.utils.data import DataLoader
    tta_loader = DataLoader(tta_set, batch_size=max(1, args.batch_size // 2), shuffle=False,
                            num_workers=0, pin_memory=True)
    for name, loader, use_tta in (
        ("test", test_loader, False),
        ("test_tta", tta_loader, True),
    ):
        logits, labels = predict_logits(final, loader, device, tta=use_tta)
        report[name] = metrics_from_logits(logits, labels, classes)
        bias = tune_logit_bias(val_logits, val_labels, classes)
        report[name + "_calibrated"] = metrics_from_logits(logits, labels, classes, bias)
        np.save(out_dir / f"{name}_logits.npy", logits)
        np.save(out_dir / f"{name}_labels.npy", labels)
        report[name + "_bias"] = bias.tolist()

    (out_dir / "test_metrics.json").write_text(json.dumps(report, indent=2))
    for k in ("test", "test_tta", "test_calibrated", "test_tta_calibrated"):
        if k in report:
            print(k, "acc=%.4f f1=%.4f" % (report[k]["accuracy"], report[k]["macro_f1"]))


if __name__ == "__main__":
    main()

Overwriting train.py


In [19]:
%%writefile make_cache.py
"""Pre-resize the potato dataset once so each epoch does not decode 1500x1500 JPEGs.

The raw images are all 1500x1500. Training with RandomResizedCrop/affine on a
448x448 cached copy is visually indistinguishable from resizing the original,
but decoding is ~15x faster (important because the DSH sandbox forbids
DataLoader worker processes here, so --num-workers must stay 0).
"""
from __future__ import annotations

import argparse
from pathlib import Path

from PIL import Image
from tqdm import tqdm


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--source", default="data/potato_split")
    ap.add_argument("--output", default="data/potato_split_448")
    ap.add_argument("--size", type=int, default=448)
    args = ap.parse_args()

    src = Path(args.source)
    dst = Path(args.output)
    for split in ("train", "val", "test"):
        for cls_dir in sorted((src / split).iterdir()):
            if not cls_dir.is_dir():
                continue
            out_dir = dst / split / cls_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            for f in tqdm(list(cls_dir.glob("*")), desc=f"{split}/{cls_dir.name}", leave=False):
                if not f.is_file():
                    continue
                im = Image.open(f).convert("RGB")
                im = im.resize((args.size, args.size), Image.BICUBIC)
                im.save(out_dir / (f.stem + ".jpg"), quality=95)
    print("cache written to", dst)


if __name__ == "__main__":
    main()

Overwriting make_cache.py


## Chọn thí nghiệm

In [20]:
RUN = "capacity"        # "capacity" | "mixup" | "long"

CONFIGS = {
    # student growth 24 -> 2.2 M params; teacher stays 384
    "capacity": dict(growth="24", epochs="250", mixup="0.0"),
    "mixup":    dict(growth="16", epochs="250", mixup="0.2"),
    "long":     dict(growth="16", epochs="400", mixup="0.0"),
}
CFG = CONFIGS[RUN]
print("RUN =", RUN, CFG)

RUN = capacity {'growth': '24', 'epochs': '250', 'mixup': '0.0'}


In [21]:
CACHE = "/kaggle/working/potato_448"
if not Path(CACHE).exists():
    !python make_cache.py --source "$DATA_ROOT" --output "$CACHE" --size 448
DATA = CACHE
print("data:", DATA)

cache written to /kaggle/working/potato_448
data: /kaggle/working/potato_448


## Teacher @384 — dùng chung cho cả 3 thí nghiệm

Chỉ cần chạy nếu chưa có `/kaggle/working/teacher384/best.pt`.

In [22]:
import os
if not Path("/kaggle/working/teacher384/best.pt").exists():
    !python train.py --data "$DATA" --output /kaggle/working/teacher384 \
      --mode teacher --seed 42 --epochs 40 --batch-size 12 --lr 1e-3 \
      --weight-decay 0.05 --label-smoothing 0.1 --aug strong --ema 0.999 \
      --image-size 384 --num-workers 4 --track mean
else:
    print("teacher384 da co, bo qua")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 189MB/s]
teacher params: 11.276206 M
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might

## Student — chạy cấu hình đã chọn

> Add blockquote



In [30]:
import subprocess, sys
args = [sys.executable, "train.py", "--data", DATA,
        "--teacher", "/kaggle/working/teacher384/best.pt",
        "--output", f"/kaggle/working/run_{RUN}",
        "--mode", "student", "--seed", "42",
        "--epochs", CFG["epochs"], "--batch-size", "32", "--lr", "1.5e-3",
        "--weight-decay", "0.05", "--label-smoothing", "0.05",
        "--aug", "fast", "--ema", "0.999",
        "--kd", "dkd", "--feature", "map", "--lambda-hard", "1.0",
        "--lambda-feature", "0.3", "--lambda-response", "1.0",
        "--lambda-self", "0.1", "--kd-ramp", "20",
        "--growth", CFG["growth"], "--mixup", CFG["mixup"],
        "--image-size", "224", "--num-workers", "4", "--track", "mean"]
print(" ".join(args))
subprocess.run(args, check=True)

/usr/bin/python3 train.py --data /kaggle/working/potato_448 --teacher /kaggle/working/teacher384/best.pt --output /kaggle/working/run_mt --mode student --seed 42 --epochs 250 --batch-size 32 --lr 1.5e-3 --weight-decay 0.05 --label-smoothing 0.05 --aug fast --ema 0.999 --kd dkd --feature map --lambda-hard 1.0 --lambda-feature 0.3 --lambda-response 1.0 --lambda-self 0.1 --kd-ramp 20 --growth 24 --mixup 0.0 --image-size 224 --num-workers 4 --track mean


KeyboardInterrupt: 

In [40]:
!rm -rf /kaggle/working/run_mt16
!cd /content && python -u train.py \
  --data /kaggle/working/potato_448 \
  --teacher /kaggle/working/teacher384/best.pt \
  --output /kaggle/working/run_mt16 \
  --mode student --seed 42 --epochs 250 --batch-size 32 --lr 1.5e-3 \
  --weight-decay 0.05 --label-smoothing 0.05 --aug fast --ema 0.999 \
  --kd dkd --feature map --lambda-hard 1.0 --lambda-feature 0.3 \
  --lambda-response 1.0 --lambda-self 0.1 --kd-ramp 20 \
  --growth 16 --image-size 224 --num-workers 2 --track mean \
  2>&1 | tee /kaggle/working/run_mt16.log

student backbone params: 1.051374 M
{"epoch": 1, "loss": 4.110512739572769, "val_accuracy": 0.11313868613138686, "val_macro_f1": 0.09265083770171653, "val_balanced_acc": 0.08568190693235057, "seconds": 26.1}
{"epoch": 2, "loss": 4.136905141365834, "val_accuracy": 0.14963503649635038, "val_macro_f1": 0.11403389320609555, "val_balanced_acc": 0.13274099731985523, "seconds": 50.8}
{"epoch": 3, "loss": 4.134383057936644, "val_accuracy": 0.20437956204379562, "val_macro_f1": 0.14589242743621764, "val_balanced_acc": 0.17260514876004387, "seconds": 74.9}
{"epoch": 4, "loss": 4.090685343131041, "val_accuracy": 0.33941605839416056, "val_macro_f1": 0.2472174660456852, "val_balanced_acc": 0.3231383690105865, "seconds": 98.2}
{"epoch": 5, "loss": 4.053452601799598, "val_accuracy": 0.3795620437956204, "val_macro_f1": 0.31167122222265375, "val_balanced_acc": 0.3956513359944346, "seconds": 123.0}
{"epoch": 6, "loss": 4.020348117901729, "val_accuracy": 0.3905109489051095, "val_macro_f1": 0.3344980677411

In [39]:
!ps aux | grep "[t]rain.py"
!tail -n 3 /kaggle/working/run_mt16.log

    )
    ^
KeyboardInterrupt


In [43]:
from pathlib import Path
for s in ("train","val","test"):
    p = Path(DATA_ROOT)/s
    print(s, sum(len(list((p/c).glob("*"))) for c in p.iterdir() if c.is_dir()))

train 2487
val 274
test 315


## Kết quả + chi phí (để báo cáo Pareto)

In [41]:
import json, torch
from torch.utils.flop_counter import FlopCounterMode
from models import HybridStudent

r = json.loads(Path(f"/kaggle/working/run_{RUN}/test_metrics.json").read_text())
m = HybridStudent(7, growth_rate=int(CFG["growth"])).eval()
with FlopCounterMode(display=False) as fc:
    with torch.no_grad():
        m(torch.randn(1, 3, 224, 224))

print(f"RUN={RUN}")
print(f"  acc      {r['test']['accuracy']*100:.2f}")
print(f"  macroF1  {r['test']['macro_f1']*100:.2f}")
print(f"  best_ep  {r['best_epoch']}")
print(f"  params   {sum(p.numel() for p in m.parameters())/1e6:.3f} M")
print(f"  GFLOPs   {fc.get_total_flops()/2e9:.3f}")
print()
print("so sanh: paper student 84.57 | ours growth16 89.83 (1.051 M, 0.630 GFLOPs)")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/run_mt/test_metrics.json'